# Exploratory Data Analysis (EDA): Fruit and Vegetable Dataset

This notebook provides a systematic exploratory analysis of the **36-class Fruit and Vegetable Image Recognition** dataset.

### Notebook Objectives
1. Verify dataset structure across training, validation, and test splits
2. Quantify per-category sample distribution and class balance
3. Generate a multi-category produce visual gallery across all 36 classes
4. Profile image resolutions, dimensions, and aspect ratio variations
5. Analyze RGB channel statistics (mean and standard deviation)
6. Inspect the data augmentation pipeline effects visually prior to model training


In [ ]:
# Environment Setup and Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from torchvision import transforms

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
print('Dependencies loaded successfully.')


## 1. Dataset Directory Setup
Locate dataset splits or automatically download via `kagglehub` if running in an isolated environment (such as Google Colab).


In [ ]:
data_dir = Path('../data') if Path('../data').exists() else Path('data')

# If data is not present locally, download via kagglehub
if not (data_dir / 'train').exists():
    print('Dataset not detected locally. Downloading via kagglehub...')
    import kagglehub
    download_path = kagglehub.dataset_download('kritikseth/fruit-and-vegetable-image-recognition')
    data_dir = Path(download_path)

train_dir = data_dir / 'train'
val_dir   = data_dir / 'validation'
test_dir  = data_dir / 'test'

print(f'Train directory     : {train_dir} (Exists: {train_dir.exists()})')
print(f'Validation directory: {val_dir} (Exists: {val_dir.exists()})')
print(f'Test directory      : {test_dir} (Exists: {test_dir.exists()})')


## 2. Split Summary and Class Distribution Analysis
Quantify total samples per split and check balance across all 36 categories.


In [ ]:
def get_class_counts(split_path):
    counts = {}
    for d in sorted(split_path.iterdir()):
        if d.is_dir():
            imgs = [f for f in d.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]
            counts[d.name] = len(imgs)
    return counts

train_counts = get_class_counts(train_dir)
val_counts   = get_class_counts(val_dir)
test_counts  = get_class_counts(test_dir)

df_splits = pd.DataFrame({
    'Train': train_counts,
    'Validation': val_counts,
    'Test': test_counts
})
df_splits['Total'] = df_splits.sum(axis=1)

print(f'Total Classes: {len(df_splits)}')
print(f'Total Training Images  : {df_splits["Train"].sum():,}')
print(f'Total Validation Images: {df_splits["Validation"].sum():,}')
print(f'Total Test Images      : {df_splits["Test"].sum():,}')
print(f'Overall Dataset Size   : {df_splits["Total"].sum():,}')

df_splits.head(10)


In [ ]:
# Visualize Class Distribution across splits
fig, ax = plt.subplots(figsize=(16, 7), dpi=150)
df_sorted = df_splits.sort_values(by='Train', ascending=False)

x = np.arange(len(df_sorted))
width = 0.6

ax.bar(x, df_sorted['Train'], width, label='Train', color='#3B82F6', alpha=0.9)
ax.bar(x, df_sorted['Validation'], width, bottom=df_sorted['Train'], label='Validation', color='#10B981', alpha=0.9)
ax.bar(x, df_sorted['Test'], width, bottom=df_sorted['Train'] + df_sorted['Validation'], label='Test', color='#F59E0B', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(df_sorted.index, rotation=60, ha='right', fontsize=9)
ax.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
ax.set_title('Dataset Distribution Across 36 Produce Categories', fontsize=14, fontweight='bold')
ax.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.show()


## 3. Produce Sample Gallery (36 Categories Grid)
Visual inspection of representative training images across all 36 classes.


In [ ]:
classes = sorted(list(train_counts.keys()))
rows, cols = 6, 6
fig, axes = plt.subplots(rows, cols, figsize=(18, 18), dpi=120)
axes = axes.flatten()

for idx, cls_name in enumerate(classes):
    cls_folder = train_dir / cls_name
    imgs = [f for f in cls_folder.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]
    if imgs:
        img = Image.open(imgs[0]).convert('RGB')
        axes[idx].imshow(img)
        axes[idx].set_title(f'{cls_name.title()}\n({len(imgs)} imgs)', fontsize=9, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Representative Gallery of 36 Produce Categories', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## 4. Image Dimensions, Resolutions, and Aspect Ratios
Profile image dimensional variations across the dataset to inform preprocessing.


In [ ]:
widths, heights, aspect_ratios = [], [], []

for cls_dir in train_dir.iterdir():
    if cls_dir.is_dir():
        for img_path in list(cls_dir.glob('*.*'))[:15]:
            try:
                with Image.open(img_path) as im:
                    w, h = im.size
                    widths.append(w)
                    heights.append(h)
                    aspect_ratios.append(w / h)
            except Exception:
                continue

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
sns.histplot(widths, bins=30, color='#2563EB', kde=True, ax=ax1, label='Width')
sns.histplot(heights, bins=30, color='#DC2626', kde=True, ax=ax1, label='Height')
ax1.set_title('Image Resolution Distribution (Pixels)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Dimension (px)')
ax1.legend()

sns.histplot(aspect_ratios, bins=30, color='#059669', kde=True, ax=ax2)
ax2.axvline(1.0, color='black', linestyle='--', label='1:1 Square Ratio')
ax2.set_title('Aspect Ratio Distribution (Width / Height)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Aspect Ratio')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Median Resolution: {int(np.median(widths))} x {int(np.median(heights))} px')
print(f'Mean Aspect Ratio: {np.mean(aspect_ratios):.2f}')


## 5. RGB Channel Statistics
Compute empirical mean and standard deviation across image channels to compare with standard ImageNet normalization.


In [ ]:
sample_files = []
for cls_dir in train_dir.iterdir():
    if cls_dir.is_dir():
        sample_files.extend(list(cls_dir.glob('*.*'))[:10])

channel_means, channel_stds = [], []
for p in sample_files:
    try:
        arr = np.array(Image.open(p).convert('RGB')) / 255.0
        channel_means.append(arr.mean(axis=(0, 1)))
        channel_stds.append(arr.std(axis=(0, 1)))
    except Exception:
        continue

empirical_mean = np.mean(channel_means, axis=0)
empirical_std  = np.mean(channel_stds, axis=0)

print('Dataset Channel Statistics vs ImageNet Baseline:')
print(f'Dataset Mean : R={empirical_mean[0]:.4f}, G={empirical_mean[1]:.4f}, B={empirical_mean[2]:.4f}')
print('ImageNet Mean: R=0.4850, G=0.4560, B=0.4060')
print(f'Dataset Std  : R={empirical_std[0]:.4f}, G={empirical_std[1]:.4f}, B={empirical_std[2]:.4f}')
print('ImageNet Std : R=0.2290, G=0.2240, B=0.2250')


## 6. Training Augmentation Visualizer
Demonstrate the stochastic transformations applied during model training:
- `RandomResizedCrop(224, scale=(0.7, 1.0))`
- `RandomHorizontalFlip(p=0.5)`
- `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)`
- `RandomRotation(degrees=15)`


In [ ]:
# Select sample image
sample_cls = classes[1]  # Banana
sample_img_path = next((train_dir / sample_cls).glob('*.*'))
orig_pil = Image.open(sample_img_path).convert('RGB')

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=15),
])

fig, axes = plt.subplots(1, 5, figsize=(16, 4), dpi=150)
axes[0].imshow(orig_pil)
axes[0].set_title('Original Input', fontweight='bold')
axes[0].axis('off')

for i in range(1, 5):
    aug_pil = train_transform(orig_pil)
    axes[i].imshow(aug_pil)
    axes[i].set_title(f'Augmentation #{i}', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Variations on Single Produce Sample', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Key Findings and Modeling Decisions
- **Balanced Benchmark:** The test split contains a balanced distribution (~10 images per class), supporting reliable macro-averaged evaluation.
- **Scale & Aspect Variation:** High variance in raw image resolutions confirms that `RandomResizedCrop(224)` is essential for geometric invariance.
- **Color Alignment:** Channel statistics are closely aligned with ImageNet distributions, validating standard ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).
